In [8]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go

# 1. Datos de ejemplo: 6 meses (generan 5 pares (X_{t-1}, X_t))
fechas = ["2005-01", "2005-02", "2005-03", "2005-04", "2005-05", "2005-06"]
precios = np.array([40.0, 70.0, 35.0, 62.0, 48.0, 58.0])

df = pd.DataFrame({
    "fecha_prev": fechas[:-1],
    "fecha_curr": fechas[1:],
    "X_prev": precios[:-1],   # X_{t-1} (Abscisa)
    "X_curr": precios[1:],    # X_t     (Ordenada observada)
    "idx_prev": range(1, len(precios)),
    "idx_curr": range(2, len(precios) + 1),
})

# 2. Ajuste OLS del modelo AR(1): X_t = b + m * X_{t-1}
m, b = np.polyfit(df["X_prev"], df["X_curr"], 1)
df["X_hat"] = b + m * df["X_prev"]
df["residuo"] = df["X_curr"] - df["X_hat"]
df["abs_error"] = np.abs(df["residuo"])

# Nivel de equilibrio de largo plazo (intersección con la identidad X_t = X_{t-1})
mu_eq = b / (1 - m)

# 3. Construcción de la figura en Plotly (Modo Oscuro)
fig = go.Figure()

# Rango para la recta de regresión
x_line = np.linspace(min(precios) - 8, max(precios) + 8, 100)
y_line = b + m * x_line

# Capa 1: Recta de regresión OLS
fig.add_trace(go.Scatter(
    x=x_line,
    y=y_line,
    mode="lines",
    name=f"Recta AR(1): X'<sub>t</sub> = {b:.2f} + ({m:.2f})·X<sub>t-1</sub>",
    line=dict(color="#39FF14", width=2.5),
    hovertemplate=(
        "<b>Recta de Regresión OLS</b><br>"
        f"Ecuación: X'<sub>t</sub> = {b:.2f} + ({m:.2f})·X<sub>t-1</sub><br>"
        "Abscisa (X<sub>t-1</sub>): %{x:.2f} USD<br>"
        "Predicción (X'<sub>t</sub>): %{y:.2f} USD"
        "<extra></extra>"
    )
))

# Capa 2: Proyecciones verticales hacia el eje X_{t-1} (guías visuales)
for _, row in df.iterrows():
    y_top = max(row["X_curr"], row["X_hat"])
    fig.add_shape(
        type="line",
        x0=row["X_prev"], y0=0,
        x1=row["X_prev"], y1=y_top,
        line=dict(color="rgba(255, 255, 255, 0.22)", width=1.2, dash="dot"),
        layer="below"
    )

# Capa 3: Segmentos de residuos |X_t - X'_t| (llaves/errores en rojo)
for i, row in df.iterrows():
    fig.add_trace(go.Scatter(
        x=[row["X_prev"], row["X_prev"]],
        y=[row["X_hat"], row["X_curr"]],
        mode="lines",
        name="Residuo |X<sub>t</sub> - X'<sub>t</sub>|" if i == 0 else None,
        showlegend=(i == 0),
        line=dict(color="#FF3131", width=2.5, dash="dash"),
        hoverinfo="skip"
    ))

# Capa 4: Puntos blancos estimados sobre la recta (X_{t-1}, X'_t)
custom_hat = np.stack((
    df["fecha_prev"], df["fecha_curr"],
    df["idx_prev"], df["idx_curr"],
    df["X_curr"], df["residuo"]
), axis=-1)

fig.add_trace(go.Scatter(
    x=df["X_prev"],
    y=df["X_hat"],
    mode="markers+text",
    name="Estimado (X<sub>t-1</sub>, X'<sub>t</sub>)",
    marker=dict(color="#FFFFFF", size=10, line=dict(color="#39FF14", width=1.5)),
    text=[f"X'<sub>{int(k)}</sub>" for k in df["idx_curr"]],
    textposition="bottom left",
    textfont=dict(color="#E0E0E0", size=12),
    customdata=custom_hat,
    hovertemplate=(
        "<b>Estimación OLS: (X<sub>%{customdata[2]}</sub>, X'<sub>%{customdata[3]}</sub>)</b><br>"
        "Fechas: %{customdata[0]} → <b>%{customdata[1]}</b><br>"
        "Abscisa real (X<sub>t-1</sub>): <b>%{x:.2f} USD</b><br>"
        "Ordenada estimada (X'<sub>t</sub>): <b>%{y:.2f} USD</b><br>"
        "Valor real observado (X<sub>t</sub>): %{customdata[4]:.2f} USD<br>"
        "Residuo (X<sub>t</sub> - X'<sub>t</sub>): %{customdata[5]:+.2f} USD"
        "<extra></extra>"
    )
))

# Capa 5: Puntos verdes observados históricos (X_{t-1}, X_t)
custom_obs = np.stack((
    df["fecha_prev"], df["fecha_curr"],
    df["idx_prev"], df["idx_curr"],
    df["X_hat"], df["abs_error"]
), axis=-1)

fig.add_trace(go.Scatter(
    x=df["X_prev"],
    y=df["X_curr"],
    mode="markers+text",
    name="Observado (X<sub>t-1</sub>, X<sub>t</sub>)",
    marker=dict(color="#00FF7F", size=11, symbol="circle", line=dict(color="#0E1117", width=1.5)),
    text=[f"(X<sub>{int(j)}</sub>, X<sub>{int(k)}</sub>)" for j, k in zip(df["idx_prev"], df["idx_curr"])],
    textposition="top right",
    textfont=dict(color="#00BFFF", size=12),
    customdata=custom_obs,
    hovertemplate=(
        "<b>Par Observado: (X<sub>%{customdata[2]}</sub>, X<sub>%{customdata[3]}</sub>)</b><br>"
        "Transición: <b>%{customdata[0]}</b> → <b>%{customdata[1]}</b><br>"
        "Abscisa (X<sub>t-1</sub>): <b>%{x:.2f} USD</b><br>"
        "Ordenada (X<sub>t</sub>): <b>%{y:.2f} USD</b><br>"
        "Predicción del modelo (X'<sub>t</sub>): %{customdata[4]:.2f} USD<br>"
        "Error absoluto |X<sub>t</sub> - X'<sub>t</sub>|: <b>%{customdata[5]:.2f} USD</b>"
        "<extra></extra>"
    )
))

# 4. Estética y layout en modo oscuro
fig.update_layout(
    template="plotly_dark",
    paper_bgcolor="#0E1117",
    plot_bgcolor="#161B22",
    title=dict(
        text=(
            "<b>Plano de Fase AR(1) — Ajuste In-Sample por MCO (OLS)</b><br>"
            f"<span style='font-size:13px; color:#A0AAB5;'>"
            f"b = {b:.2f} | m = {m:.2f} | Nivel de equilibrio μ = {mu_eq:.2f} USD</span>"
        ),
        x=0.03,
        y=0.96,
        font=dict(size=18, color="#FFFFFF")
    ),
    xaxis=dict(
        title="<b>X<sub>t-1</sub> (USD) — Valor histórico rezagado</b>",
        gridcolor="rgba(255, 255, 255, 0.08)",
        zerolinecolor="rgba(255, 255, 255, 0.25)",
        tickvals=df["X_prev"],
        ticktext=[f"{val:.0f}<br><span style='font-size:10px;color:#888'>({f})</span>"
                  for val, f in zip(df["X_prev"], df["fecha_prev"])],
        range=[min(precios) - 10, max(precios) + 12]
    ),
    yaxis=dict(
        title="<b>X<sub>t</sub> (USD) — Valor contemporáneo / estimado</b>",
        gridcolor="rgba(255, 255, 255, 0.08)",
        zerolinecolor="rgba(255, 255, 255, 0.25)",
        range=[min(precios) - 10, max(precios) + 12]
    ),
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="left",
        x=0.0,
        bgcolor="rgba(14, 17, 23, 0.85)",
        bordercolor="rgba(255, 255, 255, 0.15)",
        borderwidth=1
    ),
    hovermode="closest",
    width=950,
    height=680,
    margin=dict(l=70, r=40, t=125, b=80)
)

fig.show()